In [1]:
%pip install langchain langchain-core langchain-community langchain-text-splitters pypdf pymupdf sentence-transformers chromadb scikit-learn langchain-groq langchain-openai python-dotenv

In [44]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

for key_name in ("GROQ_API_KEY", "OPENAI_API_KEY"):
    key_value = os.getenv(key_name)
    if not key_value or key_value.startswith("your-"):
        print(f"Warning: {key_name} is missing or still uses a placeholder.")


In [2]:
# %pip install "protobuf<7,>=3.20"

In [3]:
from langchain_core.documents import Document

In [4]:
sample_doc = Document(
    page_content = "Hello World",
    metadata = {"source" : "https://www.google.com"}
)

In [5]:
sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='Hello World')

In [6]:
type(sample_doc)

langchain_core.documents.base.Document

In [7]:
# text data
from langchain_community.document_loaders.text import TextLoader

loader = TextLoader("data/Python.txt", encoding = "cp1252")

C:\Users\Siri Yasha Dinesh\AppData\Local\Temp\ipykernel_9852\2605423986.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


In [8]:
document = loader.load()

In [9]:
document

[Document(metadata={'source': 'data/Python.txt'}, page_content='Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, and mo

In [10]:
# # PDF data
# from langchain_community.document_loaders.pdf import PyPDFLoader

# pdf_loader = PyPDFLoader("data/research2.pdf")

# document = pdf_loader.load()
# document

In [11]:
# from langchain_community.document_loaders.pdf import PyMuPDFLoader # use for complex pdfs

# pdf_loader = PyMuPDFLoader("data/research.pdf")

# document = pdf_loader.load()
# document

## Ingestion Pipeline

### documents

In [12]:
# Data <=> Documents
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

In [13]:
def load_all_pdfs():
    folder_path = "data/pdfs"
    num_docs=0
    all_docs=[]

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            pdf_path = os.path.join(folder_path,filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()
            
            all_docs.extend(doc)
            num_docs+=1
    print("total pdfs : ", num_docs)
    print("total pages : ", len(all_docs))
    return all_docs

In [14]:
all_pdf_docs = load_all_pdfs()

total pdfs :  2
total pages :  32


In [15]:
all_pdf_docs[1]

Document(metadata={'producer': 'pdfcpu v0.12.1 dev', 'creator': 'PyPDF', 'creationdate': '2026-08-20T08:59:55+00:00', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'book': 'Advances in Neural Information Processing Systems 30', 'created': '2017', 'date': '2017', 'description': 'Paper accepted and presented at the Neural Information Processing Systems Conference (http://nips.cc/)', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more paralleli

In [16]:
type(all_pdf_docs[0])

langchain_core.documents.base.Document

### Chunks

In [17]:

# %pip install langchain_text_splitters

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents,chunk_size=500,chunk_overlap=50):
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size= chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [19]:
chunks = split_docs(all_pdf_docs)

In [20]:
len(chunks)

321

In [21]:
chunks

[Document(metadata={'producer': 'pdfcpu v0.12.1 dev', 'creator': 'PyPDF', 'creationdate': '2026-08-20T08:59:55+00:00', 'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin', 'book': 'Advances in Neural Information Processing Systems 30', 'created': '2017', 'date': '2017', 'description': 'Paper accepted and presented at the Neural Information Processing Systems Conference (http://nips.cc/)', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallel

### Vector Embeddings

In [22]:
from sentence_transformers import SentenceTransformer

In [23]:
class EmbeddingManager:
    def __init__(self,model_name="all-MiniLM-L6-v2"): # a model that generates embeddings
        
        self.model_name = model_name
        print("loading model .... ", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions", self.model.get_sentence_embedding_dimension())

    def generate_embeddings(self,text):
        embeddings = self.model.encode(text,show_progress_bar=True)
        print("embeddings shape : ", embeddings.shape)
        return embeddings

In [24]:
embedding_manager = EmbeddingManager()

loading model ....  all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions 384


C:\Users\Siri Yasha Dinesh\AppData\Local\Temp\ipykernel_9852\137002727.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions", self.model.get_sentence_embedding_dimension())


### Vector Store 
##### is a small scale local version of a vector DB (for small projects)

In [25]:
import chromadb
import uuid

In [26]:
class VectorStoreManager():
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.persist_directory= persist_directory
        self.collection_name = collection_name
        self.collection = None
        self.client = None

        self._initialize_store()
        
    def _initialize_store(self):
        os.makedirs(self.persist_directory,exist_ok=True)
        # create a client 
        self.client = chromadb.PersistentClient(path = self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(
            name = self.collection_name,
            metadata = {"description" : "Vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection = ",self.collection)
        print("docs in collection : ", self.collection.count())
        
    def add_docs(self,documents,embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match w the num of embeddings")

        # in collection: store => ids, embeddings, document, metadata
        ids=[]
        all_metadata = []
        documents_content=[]
        embeddings_list=[]

        for i,(doc,embedding) in enumerate(zip(documents, embeddings)): # zip(documents, embeddings) pairs up each document with its corresponding embedding, e.g., (doc1, emb1), (doc2, emb2), ...
            doc_id=f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        self.collection.add(
            ids=ids,
            metadatas=all_metadata,
            documents=documents_content,
            embeddings=embeddings_list
        )

        print("total documents added in vector store = ", len(documents_content))
        print("docs in collection : ", self.collection.count())

In [27]:
vector_store = VectorStoreManager()

initialized the vector store with collection =  Collection(name=pdf_documents)
docs in collection :  1284


In [28]:
text = [doc.page_content for doc in chunks]

embeddings = embedding_manager.generate_embeddings(text)

vector_store.add_docs(chunks,embeddings)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

embeddings shape :  (321, 384)
total documents added in vector store =  321
docs in collection :  1605


# Retrieval Pipeline

In [29]:
from sklearn.metrics.pairwise import cosine_similarity

In [30]:
class RAGRetriever:
    def __init__(self,embedding_manager,vector_store):
        self.embedding_manager=embedding_manager
        self.vector_store=vector_store

    def retrieve(self,query,top_k=5,score_threshold=0.1):
        # query to embeddings
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        #retrieve data from the vector store
        #semantic search
        results = self.vector_store.collection.query(
            query_embeddings = [query_embeddings.tolist()],
            n_results = top_k
        )

        #cosine similarity
        retrieved_docs=[]
        if results["documents"] and results["documents"][0]:
            ids=results["ids"][0]
            metadatas=results["metadatas"][0]
            documents=results["documents"][0]
            distances=results["distances"][0]

            for i,(doc_id,metadata,document,distance) in enumerate(zip(ids,metadatas,documents,distances)):
                similarity_score=1-distance
                if similarity_score>=score_threshold:
                    retrieved_docs.append({
                        "id":doc_id,
                        "document":document,
                        "metadata":metadata,
                        "similarity_score":similarity_score,
                        "rank":i+1
                    })

            print(f"retireved {len(retrieved_docs)} documents")
        else:
            print("no documents found")

        return retrieved_docs # this is our context

In [31]:
rag_retriever = RAGRetriever(embedding_manager,vector_store)

In [32]:
rag_retriever.retrieve("What is RAG?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape :  (1, 384)
retireved 5 documents


[{'id': 'doc_596a9527-089d-4b2a-8f4f-3acfdcfe552e',
  'document': 'and speculate on upcoming trends and innovations.\nOur contributions are as follows:\n• In this survey, we present a thorough and systematic\nreview of the state-of-the-art RAG methods, delineating\nits evolution through paradigms including naive RAG,\narXiv:2312.10997v5  [cs.CL]  27 Mar 2024',
  'metadata': {'creationdate': '2024-03-28T00:54:45+00:00',
   'doc_index': 88,
   'trapped': '/False',
   'content_length': 288,
   'page': 0,
   'producer': 'pdfTeX-1.40.25',
   'moddate': '2024-03-28T00:54:45+00:00',
   'total_pages': 21,
   'subject': '',
   'page_label': '1',
   'author': '',
   'source': 'data/pdfs\\research2.pdf',
   'title': '',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'keywords': '',
   'creator': 'LaTeX with hyperref'},
  'similarity_score': 0.5370849072933197,
  'rank': 1},
 {'id': 'doc_e6d1ab88-b0b7-49e1-bd9e-dcfcb815c37f',
  'd

In [33]:
rag_retriever.retrieve("What is encoder, decoder?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape :  (1, 384)
retireved 5 documents


[{'id': 'doc_40afb454-03cd-42a4-b358-0a2924dc5601',
  'document': 'layers, produce outputs of dimensiondmodel = 512.\nDecoder: The decoder is also composed of a stack ofN = 6 identical layers. In addition to the two\nsub-layers in each encoder layer, the decoder inserts a third sub-layer, which performs multi-head\nattention over the output of the encoder stack. Similar to the encoder, we employ residual connections\naround each of the sub-layers, followed by layer normalization. We also modify the self-attention',
  'metadata': {'creator': 'PyPDF',
   'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin',
   'moddate': '2026-08-20T08:59:55+00:00',
   'content_length': 447,
   'firstpage': '5998',
   'language': 'en-US',
   'created': '2017',
   'book': 'Advances in Neural Information Processing Systems 30',
   'published': '2017',
   'editors': 'I. Guyon and U.V. Luxburg and S. Bengio and H. Wallach and R. 

## Generation

## Groq

In [45]:
API_KEY_GROQ = os.environ["GROQ_API_KEY"]

In [46]:
%pip install langchain-groq

In [47]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key = API_KEY_GROQ,
    model = "qwen/qwen3.6-27b",
    temperature=0.01,
    max_tokens=None, 
)

In [48]:
def generate_output(query,retriever,llm,top_k=3):
    results = retriever.retrieve(query,top_k)

    context = "\n".join([doc["document"] for doc in results ]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    prompt = f"""use the given context to generate the answer for the query
                context : {context}
                query : {query}"""

    response = llm.invoke([prompt.format(context = context, query = query)])
    return response.content

In [49]:
answer = generate_output("what is encoder-decoder?",rag_retriever,llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape :  (1, 384)
retireved 3 documents


In [50]:
print(answer)


<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Context:** The provided text describes the "Decoder" part of a model (likely the Transformer architecture, given the mention of `dmodel = 512`, `N = 6 identical layers`, multi-head attention, residual connections, layer normalization). It's repeated three times with slight truncation at the end.
   - **Query:** "what is encoder-decoder?"
   - **Task:** Use the given context to generate an answer for the query.

2.  **Evaluate Context vs. Query:**
   - The context *only* describes the **Decoder** component. It mentions:
     - Stack of N=6 identical layers
     - Two sub-layers (like encoder) + a third sub-layer for multi-head attention over encoder output
     - Residual connections and layer normalization
     - Modified self-attention layers
     - Output dimension `dmodel = 512`
   - The context does *not* explicitly define or describe the "encoder-decoder" architecture as a whole, nor does it describe the encod

## Open-AI

In [51]:
API_KEY_OPENAI = os.environ["OPENAI_API_KEY"]

In [52]:
%pip install langchain-openai 

Note: you may need to restart the kernel to use updated packages.


In [53]:
# Set OPENAI_API_KEY in your environment before running this notebook.

In [54]:

from langchain_openai import ChatOpenAI

llm_openai = ChatOpenAI(
    openai_api_key = API_KEY_OPENAI,
    model = "gpt-4o-mini",
    temperature=0.01,
    max_tokens=1024
)

In [ ]:
def generate_output_OPENAI(query,retriever,llm,top_k=3):
    results = retriever.retrieve(query,top_k)

    context = "\n".join([doc["document"] for doc in results ]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    prompt = f"""use the given context to generate the answer for the query
                context : {context}
                query : {query}"""

    response = llm_openai.invoke(prompt)
    return response.content

In [ ]:
answer = generate_output_OPENAI("what is encoder-decoder?",rag_retriever,llm_openai)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape :  (1, 384)
retireved 3 documents


In [57]:
print(answer)

An encoder-decoder is a neural network architecture commonly used in tasks such as machine translation, text summarization, and other sequence-to-sequence tasks. It consists of two main components: the encoder and the decoder.

1. **Encoder**: The encoder processes the input data and compresses it into a fixed-size context representation. It typically consists of multiple layers (in this case, 6 identical layers), each containing sub-layers that perform operations like self-attention and feed-forward neural networks. The encoder outputs a representation of the input data, which captures its essential features.

2. **Decoder**: The decoder takes the context representation produced by the encoder and generates the output sequence. It also consists of multiple layers (6 identical layers in this case) and includes an additional sub-layer that performs multi-head attention over the encoder's output. This allows the decoder to focus on different parts of the input sequence while generating t